## Functional enrichment analysis

### Overview
To interpret the biological significance of the differentially expressed genes,
a functional enrichment analysis was performed using **g:Profiler**.

Enrichment was computed separately for **up-regulated** and **down-regulated** genes
to capture direction-specific functional patterns.

### Gene selection
For each differential expression result, genes were filtered based on:
- adjusted p-value < 0.05
- absolute log fold change > 0.5

Up- and down-regulated gene sets were analyzed independently to avoid mixing
opposing biological effects.

### Enrichment analysis
Functional enrichment was performed against multiple annotation sources, including:
- **Gene Ontology – Biological Process (GO:BP)**
- **Gene Ontology – Cellular Component (GO:CC)**
- **KEGG pathways**

The analysis was carried out using the human genome as reference.

### Focus on Cellular Component
While multiple annotation categories were explored, particular attention was given
to **Gene Ontology – Cellular Component (GO:CC)** terms.


Cellular Component enrichment provides a spatial perspective on transcriptional
changes, complementing pathway- and process-based interpretations.

### Output
For each comparison and regulation direction, enrichment results were stored as
structured tables for downstream filtering, visualization, and interpretation.


In [ ]:
import os
import pandas as pd
from gprofiler import GProfiler
from pathlib import Path
import yaml

cfg = yaml.safe_load(open("configs/config.yaml"))

input_dir = Path(cfg["paths"]["input_gene_expression"])
output_dir = Path(cfg["paths"]["outdir_gprofiler"])
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
!pip install gprofiler-official pandas xlsxwriter openpyxl

In [ ]:
gp = GProfiler(return_dataframe=True)

def run_enrichment(df, direction):
    if direction == "Up":
        filtered = df[(df["logfoldchanges"] > 0.5) & (df["pvals_adj"] < 0.05)]
    elif direction == "Down":
        filtered = df[(df["logfoldchanges"] < -0.5) & (df["pvals_adj"] < 0.05)]
    else:
        raise ValueError("direction must be 'Up' or 'Down'")

    query_genes = filtered["Gene_Name"].dropna().unique().tolist()
    if len(query_genes) == 0:
        print(f"No significant {direction}-regulated genes found.")
        return None

    results = gp.profile(
        organism="hsapiens",
        query=query_genes,
        sources=["GO:BP", "GO:CC", "KEGG"],
        no_evidences=False
    )
    return results


In [ ]:
source_map = {"GO:BP": "GO_BP", "GO:CC": "GO_CC", "KEGG": "KEGG"}

for file in os.listdir(input_dir):
    if file.endswith(".xlsx"):
        filepath = os.path.join(input_dir, file)
        print(f"Processing {file}")

        df = pd.read_excel(filepath)
        required_cols = {"logfoldchanges", "pvals_adj", "Gene_Name"}
        if not required_cols.issubset(df.columns):
            print(f"Missing columns in {file}: {required_cols}")
            continue

        results_up = run_enrichment(df, "Up")
        if results_up is not None:
            out_up = os.path.join(output_dir, file.replace(".xlsx", "_Upregulated.xlsx"))
            with pd.ExcelWriter(out_up, engine="xlsxwriter") as writer:
                for source, sheetname in source_map.items():
                    subset = results_up[results_up["source"] == source]
                    subset.to_excel(writer, index=False, sheet_name=sheetname)
            print(f"Saved {out_up}")

        results_down = run_enrichment(df, "Down")
        if results_down is not None:
            out_down = os.path.join(output_dir, file.replace(".xlsx", "_Downregulated.xlsx"))
            with pd.ExcelWriter(out_down, engine="xlsxwriter") as writer:
                for source, sheetname in source_map.items():
                    subset = results_down[results_down["source"] == source]
                    subset.to_excel(writer, index=False, sheet_name=sheetname)
            print(f"Saved {out_down}")

print("Done.")

## GO:CC enrichment bubble plots (renal cell populations)

### Purpose
To summarize and compare **Cellular Component (GO:CC)** enrichment results across contrasts, we generated bubble plots restricted to **renal cell types only**. The goal of this figure is to highlight how specific subcellular compartments are enriched in **some contrasts but not others**, and to support these patterns using both **significance** and the **number of genes contributing to each term**.

### Input data
For each **contrast × cell type × regulation direction (Up/Down)**, enrichment results were loaded from g:Profiler output files. Only the **GO:CC** sheet was used.

### What is shown in the plot
Each bubble represents one GO:CC term enriched in a given cell type (and direction).

- **x-axis:** renal cell type  
- **y-axis:** GO:CC term  
- **Bubble color:** enrichment significance, shown as **−log10(adjusted p-value)**  
- **Bubble size:** **number of genes** overlapping the GO:CC term (intersection size)

Non-significant terms (adjusted p-value > 0.05) are shown in **light gray**, to visually separate them from significant enrichments.

### Common scale across plots
To enable direct visual comparison across contrasts and directions, a **common color scale** and a **common bubble-size scale** were used:
- the color scale (−log10 adjusted p-value) was fixed using the global maximum across all significant results;
- the bubble-size scaling was fixed using the global maximum number of overlapping genes.

This ensures that differences between plots reflect real differences in enrichment strength and support, rather than plot-specific scaling artifacts.

### Selected GO:CC terms
To focus the figure on compartments of interest and improve interpretability, enrichment was optionally restricted to curated sets of GO:CC terms (separately for **Upregulated** and **Downregulated** gene sets). This reduces redundancy and emphasizes biologically relevant compartments.

### Interpretation guide
This visualization is intended to answer questions such as:
- Which **subcellular compartments** are consistently enriched across renal cell types?
- Which GO:CC signals are **contrast-specific** (present in one comparison but absent in others)?
- Are the most significant terms also supported by a **large number of genes**, or are they driven by small gene sets?




In [ ]:

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import yaml


from pathlib import Path
import yaml

cfg = yaml.safe_load(open("configs/config.yaml"))

DATA_DIR = Path(cfg["paths"]["gprofiler_renal_xlsx_dir"])
OUT_DIR  = Path(cfg["paths"]["gprofiler_renal_fig_dir"])

PATTERN = "*.xlsx"
SHEET_INDEX = 1  # 2nd sheet (CC)

ONLY_SOURCE = "GO:CC"
ONLY_CONTRAST = None
TOP_N_PER_GROUP = None

OUT_PREFIX = "CC_bubbleplot"

# GO termini diversi per Up e Down
GO_KEEP_UP = {
    "lysosome",
    "mitochondrial envelope",
    "apical plasma membrane",
}

GO_KEEP_DOWN = {
    "extracellular matrix",
    "nuclear chromosome",
    "chromosomal region",
    "apical plasma membrane",
    "cell-cell junction",
    "plasma membrane protein complex",
    "cell surface",
}

FILTER_TO_GO_KEEP = True

# - Common Scale 
ALPHA_SIG = 0.05


# PARSE FILENAME

def parse_filename(path):
    """
    KO_IB_vs_KO_Non-treated__Distal_Nephron_Epithelial_Cell_Downregulated.xlsx
    """
    fname = os.path.basename(path)
    m = re.match(r"^(?P<contrast>.+?)__(?P<cell>.+)_(?P<direction>Downregulated|Upregulated)\.xlsx$", fname)
    if not m:
        raise ValueError(f"Filename non matcha il pattern atteso: {fname}")

    contrast_raw = m.group("contrast")
    cell_raw = m.group("cell")
    direction = m.group("direction")

    cell_type = cell_raw.replace("_", " ")
    contrast_clean = contrast_raw.replace("_", " ")

    return contrast_raw, contrast_clean, cell_type, direction

def safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)

def norm_term(s: str) -> str:
    s = str(s).strip().lower()
    s = s.replace("–", "-").replace("—", "-").replace("‐", "-")
    s = re.sub(r"\s+", " ", s)
    return s

# Bubble
def bubble_plot(
    df,
    title,
    outpath=None,
    alpha_sig=0.05,
    nonsig_color="lightgray",
    fig_w=None,
    fig_h=None,
    base_size=12,
    size_range=220,
    x_offset=0.16,

    color_vmin=None,
    color_vmax=None,
    size_smax=None,
    size_legend_vals=None,
):
    df = df.copy()

    group_cols = ["cell_type", "term"]
    use_direction = ("direction" in df.columns) and (df["direction"].nunique() > 1)
    if use_direction:
        group_cols.append("direction")

    df = (df.groupby(group_cols, as_index=False)
            .agg(n_genes=("n_genes", "max"),
                 p_value=("p_value", "min")))

    df["neglog10_p"] = (-np.log10(df["p_value"].clip(lower=1e-300))).clip(lower=0)
    df["is_sig"] = df["p_value"] <= alpha_sig

    cell_order = sorted(df["cell_type"].unique())
    term_order = sorted(df["term"].unique())
    x_map = {c: i for i, c in enumerate(cell_order)}
    y_map = {t: i for i, t in enumerate(term_order)}

    x = df["cell_type"].map(x_map).astype(float).to_numpy()
    y = df["term"].map(y_map).astype(int).to_numpy()

    # Offset Up/Down
    if use_direction:
        offsets = {"Downregulated": -x_offset, "Upregulated": +x_offset}
        x = x + df["direction"].map(offsets).fillna(0).to_numpy()

    
    sizes = pd.to_numeric(df["n_genes"], errors="coerce").to_numpy(dtype=float)
    sizes_pos = sizes[np.isfinite(sizes) & (sizes > 0)]
    local_smax = sizes_pos.max() if sizes_pos.size else 0.0
    smax_use = float(size_smax) if (size_smax is not None and size_smax > 0) else float(local_smax)

    s = base_size + (np.nan_to_num(sizes, nan=0.0) / smax_use * size_range if smax_use > 0 else 0)

    if fig_w is None:
        fig_w = max(9.0, 0.55 * len(cell_order) + 3.44)
    if fig_h is None:
        fig_h = 4

    fig = plt.figure(figsize=(fig_w, fig_h), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 0.33])
    ax = fig.add_subplot(gs[0, 0])
    axR = fig.add_subplot(gs[0, 1])
    axR.axis("off")

    m_nonsig = (~df["is_sig"]).to_numpy()
    if m_nonsig.any():
        ax.scatter(
            x[m_nonsig], y[m_nonsig],
            s=s[m_nonsig],
            c=nonsig_color,
            alpha=0.65,
            linewidths=0
        )

    m_sig = (df["is_sig"]).to_numpy()
    if m_sig.any():
        vmin_use = 0.0 if color_vmin is None else float(color_vmin)
        vmax_use = None if color_vmax is None else float(color_vmax)

        sc_sig = ax.scatter(
            x[m_sig], y[m_sig],
            s=s[m_sig],
            c=df.loc[m_sig, "neglog10_p"].to_numpy(dtype=float),
            alpha=0.80,
            linewidths=0,
            vmin=vmin_use,
            vmax=vmax_use
        )

        cax = axR.inset_axes([0.10, 0.52, 0.35, 0.40])
        cbar = fig.colorbar(sc_sig, cax=cax)
        cbar.set_label(r"$-\log_{10}(p_{\mathrm{adj}})$")
        cbar.set_ticks([2,6,10])

    ax.set_xticks(range(len(cell_order)))
    ax.set_xticklabels(cell_order, rotation=35, ha="right")
    ax.set_yticks(range(len(term_order)))
    ax.set_yticklabels(term_order)
    ax.invert_yaxis()

    ax.set_xlabel("Cell type")
    ax.set_ylabel("GO:CC term")
    ax.set_title(title)

    pad = 0.05
    extra = (x_offset + 0.10) if use_direction else 0.10
    ax.set_xlim(-0.5 - extra - pad, (len(cell_order) - 0.5) + extra + pad)
    ax.set_ylim(-0.5 - pad, (len(term_order) - 0.5) + pad)

    ax.tick_params(axis="x", pad=2, labelsize=9)
    ax.tick_params(axis="y", pad=1, labelsize=9)

    if smax_use > 0:
        if size_legend_vals is not None:
            q = np.array(size_legend_vals, dtype=float)
            q = q[np.isfinite(q) & (q > 0)]
            q = np.unique(np.round(q).astype(int))
        else:
            q = np.quantile(sizes_pos, [0.25, 0.5, 0.75])
            q = np.unique(np.round(q).astype(int))
            if q.size < 3 and sizes_pos.size:
                q = np.unique(np.round(np.array([sizes_pos.min(),
                                                 np.median(sizes_pos),
                                                 sizes_pos.max()])).astype(int))

        size_handles = [
            ax.scatter([], [], s=base_size + (val / smax_use) * size_range, alpha=0.75, color="gray")
            for val in q
        ]
        size_labels = [f"{int(val)}" for val in q]

        axR.legend(
            size_handles, size_labels,
            title="N° geni",
            loc="upper left",
            bbox_to_anchor=(0.05, 0.48),
            frameon=True,
            fontsize=8,
            title_fontsize=9,
            borderpad=0.4,
            labelspacing=0.3,
            handletextpad=0.6
        )

    if outpath:
        fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


rows = []
files = sorted(glob.glob(os.path.join(DATA_DIR, PATTERN)))
if not files:
    raise FileNotFoundError(f"Nessun file trovato in {DATA_DIR}/{PATTERN}")

for fp in files:
    contrast_raw, contrast, cell_type, direction = parse_filename(fp)

    if ONLY_CONTRAST is not None and contrast_raw != ONLY_CONTRAST:
        continue

    df = pd.read_excel(fp, sheet_name=SHEET_INDEX, engine="openpyxl")

    needed = ["source", "name", "p_value", "intersection_size"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"In {fp} mancano colonne: {missing}. Colonne presenti: {list(df.columns)}")

    if ONLY_SOURCE is not None:
        df = df[df["source"] == ONLY_SOURCE].copy()

    out = pd.DataFrame({ 
        "contrast_raw": contrast_raw,
        "contrast": contrast,
        "cell_type": cell_type,
        "direction": direction,
        "term": df["name"].astype(str),
        "p_value": pd.to_numeric(df["p_value"], errors="coerce"),
        "n_genes": pd.to_numeric(df["intersection_size"], errors="coerce"),
    })
    rows.append(out)

data = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if data.empty:
    raise ValueError("Dopo i filtri non è rimasta nessuna riga da plottare (nessun file o filtri troppo stretti).")


data["term_norm"] = data["term"].apply(norm_term)

if FILTER_TO_GO_KEEP:
    keep_up = {norm_term(t) for t in GO_KEEP_UP}
    keep_down = {norm_term(t) for t in GO_KEEP_DOWN}

    mask_up = (data["direction"] == "Upregulated") & (data["term_norm"].isin(keep_up))
    mask_down = (data["direction"] == "Downregulated") & (data["term_norm"].isin(keep_down))

    data = data[mask_up | mask_down].copy()

if data.empty:
    raise ValueError("Dopo il filtro GO_KEEP (Up/Down) non è rimasta nessuna riga. Controlla i nomi dei termini.")

if TOP_N_PER_GROUP is not None:
    data = (
        data.sort_values("p_value", ascending=True)
            .groupby(["contrast_raw", "cell_type", "direction"], as_index=False, group_keys=False)
            .head(TOP_N_PER_GROUP)
    )

data["neglog10_p"] = (-np.log10(data["p_value"].clip(lower=1e-300))).clip(lower=0)

print("Conteggi per direction dopo filtri:")
print(data["direction"].value_counts(dropna=False))


sig_vals = data.loc[data["p_value"] <= ALPHA_SIG, "neglog10_p"].dropna()
COMMON_COLOR_VMIN = 0.0
COMMON_COLOR_VMAX = float(sig_vals.max()) if len(sig_vals) else 1.0

sizes_all = data["n_genes"].dropna()
COMMON_SMAX = float(sizes_all.max()) if len(sizes_all) else 1.0

q = np.quantile(sizes_all, [0.25, 0.5, 0.75]) if len(sizes_all) else np.array([1, 1, 1])


    
COMMON_SIZE_LEGEND = np.unique(np.round(np.array([
        sizes_all.min(),
        sizes_all.quantile(0.25),
        sizes_all.quantile(0.75),
        sizes_all.max()
    ])).astype(int))


print("COMMON_COLOR_VMAX:", COMMON_COLOR_VMAX)
print("COMMON_SMAX:", COMMON_SMAX)
print("COMMON_SIZE_LEGEND:", COMMON_SIZE_LEGEND)


os.makedirs(OUT_DIR, exist_ok=True)

def save_up_down(df_one_contrast, contrast_raw):
    # UP
    sub_up = df_one_contrast[df_one_contrast["direction"] == "Upregulated"].copy()
    if not sub_up.empty:
        title_up = f"{contrast_raw} — Upregulated — {ONLY_SOURCE or 'ALL sources'}"
        fname_up = f"01_{OUT_PREFIX}_{safe_name(contrast_raw)}_Upregulated.pdf"
        outpath_up = os.path.join(OUT_DIR, fname_up)
        bubble_plot(
            sub_up, title=title_up, outpath=outpath_up,
            alpha_sig=ALPHA_SIG,
            color_vmin=COMMON_COLOR_VMIN, color_vmax=COMMON_COLOR_VMAX,
            size_smax=COMMON_SMAX, size_legend_vals=COMMON_SIZE_LEGEND
        )
        print("Saved:", outpath_up)

    sub_down = df_one_contrast[df_one_contrast["direction"] == "Downregulated"].copy()
    if not sub_down.empty:
        title_down = f"{contrast_raw} — Downregulated — {ONLY_SOURCE or 'ALL sources'}"
        fname_down = f"02_{OUT_PREFIX}_{safe_name(contrast_raw)}_Downregulated.pdf"
        outpath_down = os.path.join(OUT_DIR, fname_down)
        bubble_plot(
            sub_down, title=title_down, outpath=outpath_down,
            alpha_sig=ALPHA_SIG,
            color_vmin=COMMON_COLOR_VMIN, color_vmax=COMMON_COLOR_VMAX,
            size_smax=COMMON_SMAX, size_legend_vals=COMMON_SIZE_LEGEND
        )
        print("Saved:", outpath_down)

if data["contrast_raw"].nunique() > 1:
    for contrast_raw, sub in data.groupby("contrast_raw", sort=True):
        save_up_down(sub, contrast_raw)
else:
    contrast_raw = data["contrast_raw"].iloc[0]
    save_up_down(data, contrast_raw)


In [ ]:

chk = (data.groupby(["contrast_raw","cell_type","direction"])
          .agg(n_terms=("term","nunique"),
               n_genes_min=("n_genes","min"),
               n_genes_max=("n_genes","max"))
          .sort_values(["contrast_raw","cell_type","direction"]))
print(chk.to_string())
